# Tourism RFM clustering reusable template

**Short name (GitHub):** `TourRFM`

Clone this notebook onto another booking extract (another charter, year, or currency). Keep the grain honest: one RFM row per guest, log1p before scale, name bins from original-unit medians.

Replace the paths and column names in section 1. Everything downstream is generic.


## Contract

| Piece | Rule |
|-------|------|
| Input grain | one booking with guest, timestamp, qty, unit amount |
| Output grain | one row per guest + Cluster + Segment |
| Snapshot | max timestamp + 1 day (or a business month-end) |
| Frequency | distinct BookingNo / order ids, never product-line count |
| Transforms | log1p → z-score; keep μ,σ next to the model |
| k | elbow + silhouette + **ops capacity** (how many treatments can Retail fund?) |
| Naming | medians in original units; never the integer that K-Means happened to emit |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import TourRFM as br

PATH = "data/tour_bookings.csv"
CUSTOMER = "GuestID"
TXN = "BookingNo"
WHEN = "StayDate"
QTY = "Quantity"
AMT = "UnitRate"
DATE_FMT = "%d.%m.%Y %H:%M"
K = 4
RANDOM_STATE = 42


In [ ]:
raw = pd.read_csv(PATH)
sales = raw.dropna(subset=[CUSTOMER]).copy()
sales = sales[sales[QTY] > 0]
sales = sales[sales[AMT] > 0]
sales["TotalSum"] = sales[QTY] * sales[AMT]
sales[WHEN] = pd.to_datetime(sales[WHEN], format=DATE_FMT)
sales[CUSTOMER] = sales[CUSTOMER].astype(int)

snap = sales[WHEN].max() + pd.Timedelta(days=1)
rfm = (
    sales.groupby(CUSTOMER)
    .agg(
        Recency=(WHEN, lambda x: (snap - x.max()).days),
        Frequency=(TXN, "nunique"),
        Monetary=("TotalSum", "sum"),
    )
    .reset_index()
)
print(len(rfm), "guest |", round(rfm["Monetary"].sum(), 2), "flow | snapshot", snap)


In [ ]:
rfm_log, X, mu, sd = tr.log_scale(rfm)
ks, inertias = tr.elbow_inertias(X, range(1, 11), random_state=RANDOM_STATE)
print("inertia", [round(float(v), 1) for v in inertias])

model, labels = tr.fit_kmeans(X, k=K)
rfm["Cluster"] = labels
print(rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].median().round(1))
print(tr.name_from_medians(rfm, labels))

Z, ev, _ = tr.pca2(X)
fig, ax = plt.subplots()
ax.scatter(Z[:, 0], Z[:, 1], c=rfm["Cluster"], s=14, alpha=0.7, cmap="tab10")
ax.set_title(f"PCA view, k={K}  ({100*float(np.sum(ev)):.1f}% in 2-D)")
plt.show()


## How to adapt

1. Confirm date format. Pandas ≥2 raises on mixed strings — set `DATE_FMT`.
2. If the ledger already has an `SessionID` / `OrderID`, use that for Frequency.
3. If Monetary can be zero (fee-only waived guest), `log1p` still works; decide whether those guest belong in the book.
4. Recompute names every time you change k or the window. Do not hard-code `Cluster==1 → core` in production without a mapping table.
5. Persist `mu`, `sd`, centres, and the name map together.
6. Do not pass the segment to a credit engine. This is a CRM grouping.
